In [1]:
import pandas as pd
df = pd.read_csv(
    "cleaned_sensor_data.csv",
    parse_dates=["time"]
)
df = df.sort_values("time").reset_index(drop=True)# Sort chronologically
print(df.shape)
print(df.dtypes)
print(df.head())

(377719, 7)
time                        datetime64[us]
Cyclone_Inlet_Gas_Temp             float64
Cyclone_Material_Temp              float64
Cyclone_Outlet_Gas_draft           float64
Cyclone_cone_draft                 float64
Cyclone_Gas_Outlet_Temp            float64
Cyclone_Inlet_Draft                float64
dtype: object
                 time  Cyclone_Inlet_Gas_Temp  Cyclone_Material_Temp  \
0 2017-01-01 00:00:00                  867.63                 910.42   
1 2017-01-01 00:05:00                  879.23                 918.14   
2 2017-01-01 00:10:00                  875.67                 924.18   
3 2017-01-01 00:15:00                  875.28                 923.15   
4 2017-01-01 00:20:00                  891.66                 934.26   

   Cyclone_Outlet_Gas_draft  Cyclone_cone_draft  Cyclone_Gas_Outlet_Temp  \
0                   -189.54             -186.04                   852.13   
1                   -184.33             -182.10                   862.53   
2           

In [ ]:
print("Missing timestamps:", df["time"].isna().sum())#verify the handoff from the previous data cleaning.
print("Duplicate timestamps:", df["time"].duplicated().sum())

Missing timestamps: 0
Duplicate timestamps: 0


In [3]:
sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Inlet_Draft",
    "Cyclone_Material_Temp"
]#Define the sensor columns

First featuring into group and finding relationships between sensors

In [4]:
# Temperature relationship features
df["Temp_Inlet_Outlet_Diff"] = (
    df["Cyclone_Inlet_Gas_Temp"]
    - df["Cyclone_Gas_Outlet_Temp"]
)
df["Temp_Material_Inlet_Diff"] = (
    df["Cyclone_Material_Temp"]
    - df["Cyclone_Inlet_Gas_Temp"]
)
df["Temp_Material_Outlet_Diff"] = (
    df["Cyclone_Material_Temp"]
    - df["Cyclone_Gas_Outlet_Temp"]
)

In [5]:
# Draft relationship features
df["Draft_Inlet_Outlet_Diff"] = (
    df["Cyclone_Inlet_Draft"]
    - df["Cyclone_Outlet_Gas_draft"]
)

df["Draft_Cone_Inlet_Diff"] = (
    df["Cyclone_cone_draft"]
    - df["Cyclone_Inlet_Draft"]
)

df["Draft_Cone_Outlet_Diff"] = (
    df["Cyclone_cone_draft"]
    - df["Cyclone_Outlet_Gas_draft"]
)

In [6]:
relationship_cols = [
    "Temp_Inlet_Outlet_Diff",
    "Temp_Material_Inlet_Diff",
    "Temp_Material_Outlet_Diff",
    "Draft_Inlet_Outlet_Diff",
    "Draft_Cone_Inlet_Diff",
    "Draft_Cone_Outlet_Diff"
]

df[relationship_cols].describe()

,Temp_Inlet_Outlet_Diff,Temp_Material_Inlet_Diff,Temp_Material_Outlet_Diff,Draft_Inlet_Outlet_Diff,Draft_Cone_Inlet_Diff,Draft_Cone_Outlet_Diff
count,376398.000000,376128.000000,376127.000000,376396.000000,376397.000000,376398.00000
mean,11.588038,23.440803,35.030061,36.518857,-23.270214,13.24820
std,44.070473,92.909899,101.912607,23.102654,24.151058,25.41559
min,-910.620000,-1131.590000,-1141.720000,-208.300000,-252.710000,-232.23000
25%,-6.350000,5.000000,4.440000,23.820000,-41.510000,-4.00000
50%,0.670000,27.140000,30.730000,44.150000,-24.830000,8.90000
75%,21.540000,48.160000,62.010000,53.740000,-1.930000,27.63000
max,805.070000,992.680000,993.820000,275.820000,583.800000,619.34000


In [7]:
# Look at extreme relationship values
for col in relationship_cols:
    print(f"\n{col}")
    print("1% :", df[col].quantile(0.01))
    print("5% :", df[col].quantile(0.05))
    print("95%:", df[col].quantile(0.95))
    print("99%:", df[col].quantile(0.99))


Temp_Inlet_Outlet_Diff
1% : -44.84999999999991
5% : -30.05150000000003
95%: 73.51999999999998
99%: 198.48029999999972

Temp_Material_Inlet_Diff
1% : -287.4576
5% : -34.64
95%: 80.22000000000003
99%: 228.7210999999987

Temp_Material_Outlet_Diff
1% : -279.97
5% : -33.6
95%: 136.31700000000018
99%: 328.3695999999996

Draft_Inlet_Outlet_Diff
1% : -6.32
5% : -3.5599999999999996
95%: 63.97
99%: 73.24000000000001

Draft_Cone_Inlet_Diff
1% : -70.64000000000001
5% : -59.75
95%: 10.789999999999992
99%: 38.330399999999806

Draft_Cone_Outlet_Diff
1% : -33.75999999999999
5% : -19.789999999999992
95%: 60.34
99%: 85.86999999999998


In [8]:
# Change from the previous 5-minute observation
for col in sensor_cols:
    df[f"{col}_change_5min"] = df[col].diff()

In [ ]:
change_5min_cols = [
    f"{col}_change_5min"
    for col in sensor_cols
]
df[change_5min_cols].describe()
'''The median change is basically 0 for every sensor:
- Most consecutive 5-minute readings don't change much.
- But the standard deviations and extreme values are much larger.'''


,Cyclone_Inlet_Gas_Temp_change_5min,Cyclone_Gas_Outlet_Temp_change_5min,Cyclone_Outlet_Gas_draft_change_5min,Cyclone_cone_draft_change_5min,Cyclone_Inlet_Draft_change_5min,Cyclone_Material_Temp_change_5min
count,376319.000000,376317.000000,376317.000000,376319.000000,376315.000000,375778.000000
mean,0.002653,0.002966,-0.001239,-0.001434,-0.001414,0.004560
std,24.203379,15.424521,9.095186,10.710795,8.388296,58.027255
min,-938.110000,-830.450000,-268.660000,-411.530000,-223.000000,-1374.010000
25%,-7.990000,-4.550000,-3.510000,-4.170000,-3.380000,-10.830000
50%,0.000000,-0.010000,-0.010000,0.000000,-0.010000,0.000000
75%,8.590000,4.710000,3.300000,4.040000,3.180000,12.360000
max,907.220000,866.930000,318.070000,729.730000,255.970000,1330.630000


In [11]:
# Change over the previous 30 minutes
for col in sensor_cols:
    df[f"{col}_change_30min"] = df[col].diff(6)

In [ ]:
change_30min_cols = [
    f"{col}_change_30min"
    for col in sensor_cols
]

df[change_30min_cols].describe()
'''he pattern is consistent:
- Median changes remain close to 0.
- The standard deviation increases because we're measuring movement over a longer window.
- Large jumps are still present.
- Cyclone_Material_Temp_change_30min is again the most volatile of the six.'''

,Cyclone_Inlet_Gas_Temp_change_30min,Cyclone_Gas_Outlet_Temp_change_30min,Cyclone_Outlet_Gas_draft_change_30min,Cyclone_cone_draft_change_30min,Cyclone_Inlet_Draft_change_30min,Cyclone_Material_Temp_change_30min
count,376143.000000,376141.000000,376141.000000,376143.000000,376139.000000,375605.000000
mean,0.015097,0.016081,-0.005589,-0.005306,-0.004974,0.019127
std,30.524351,24.740584,16.305067,16.915852,13.952893,75.441206
min,-908.200000,-858.650000,-262.320000,-486.090000,-242.670000,-1551.390000
25%,-10.910000,-8.080000,-4.450000,-5.150000,-4.150000,-15.210000
50%,-0.040000,-0.040000,-0.040000,-0.020000,-0.030000,0.000000
75%,10.620000,7.770000,3.800000,4.780000,3.620000,15.140000
max,880.000000,867.300000,397.750000,719.110000,353.860000,1363.300000


In [13]:
# Change over the previous 60 minutes
for col in sensor_cols:
    df[f"{col}_change_60min"] = df[col].diff(12)
    change_60min_cols = [
    f"{col}_change_60min"
    for col in sensor_cols
]
df[change_60min_cols].describe()

,Cyclone_Inlet_Gas_Temp_change_60min,Cyclone_Gas_Outlet_Temp_change_60min,Cyclone_Outlet_Gas_draft_change_60min,Cyclone_cone_draft_change_60min,Cyclone_Inlet_Draft_change_60min,Cyclone_Material_Temp_change_60min
count,375982.000000,375980.000000,375980.000000,375982.000000,375978.000000,375444.000000
mean,0.029862,0.033428,-0.008390,-0.008025,-0.007243,0.051711
std,34.936500,30.182189,21.252341,21.233511,17.736801,86.347516
min,-923.500000,-797.210000,-285.100000,-486.720000,-243.120000,-1375.000000
25%,-12.230000,-9.730000,-5.170000,-5.880000,-4.770000,-17.140000
50%,-0.080000,-0.070000,-0.070000,-0.030000,-0.060000,0.000000
75%,11.740000,9.280000,4.360000,5.450000,4.070000,16.950000
max,932.660000,867.300000,327.400000,712.170000,284.390000,1331.980000


adding rolling statistics

In [14]:
# 1-hour rolling statistics
for col in sensor_cols:
    df[f"{col}_rolling_mean_60min"] = (
        df[col].rolling(window=12, min_periods=6).mean()
    )
    df[f"{col}_rolling_std_60min"] = (
        df[col].rolling(window=12, min_periods=6).std()
    )

In [15]:
rolling_cols = []
for col in sensor_cols:
    rolling_cols.append(f"{col}_rolling_mean_60min")
    rolling_cols.append(f"{col}_rolling_std_60min")

In [16]:
df[rolling_cols].describe()

,Cyclone_Inlet_Gas_Temp_rolling_mean_60min,Cyclone_Inlet_Gas_Temp_rolling_std_60min,Cyclone_Gas_Outlet_Temp_rolling_mean_60min,Cyclone_Gas_Outlet_Temp_rolling_std_60min,Cyclone_Outlet_Gas_draft_rolling_mean_60min,Cyclone_Outlet_Gas_draft_rolling_std_60min,Cyclone_cone_draft_rolling_mean_60min,Cyclone_cone_draft_rolling_std_60min,Cyclone_Inlet_Draft_rolling_mean_60min,Cyclone_Inlet_Draft_rolling_std_60min,Cyclone_Material_Temp_rolling_mean_60min,Cyclone_Material_Temp_rolling_std_60min
count,376496.000000,376496.000000,376496.000000,376496.000000,376496.000000,376496.000000,376496.000000,376496.000000,376496.000000,376496.000000,376495.000000,376495.000000
mean,727.281130,14.644876,715.690412,10.461279,-177.796778,5.910509,-164.551269,6.887547,-141.284252,5.410479,750.742183,24.959505
std,328.148280,13.879469,325.050307,11.657894,98.687255,8.203010,89.532166,8.175007,77.173027,6.875346,347.735615,43.084021
min,14.450000,0.000000,13.851667,0.000000,-356.259167,0.000000,-323.558333,0.000000,-301.192500,0.000000,-185.000000,0.000000
25%,864.658958,6.665241,803.223542,4.284993,-247.065000,3.023676,-226.505208,3.743122,-193.474375,3.006719,877.926970,9.927251
50%,886.423333,12.592811,873.200000,8.392664,-215.475000,4.843813,-198.514167,5.829573,-169.589167,4.585916,917.591667,18.227612
75%,899.938333,19.814389,899.160833,13.440118,-170.508125,6.778364,-145.478542,8.077533,-136.569583,6.270021,939.679167,27.277543
max,1064.556667,439.601789,1353.188333,452.804716,30.987500,166.369547,61.350833,232.005810,30.780833,133.887163,1206.834167,645.912499


feature group on operating regime

In [17]:
def temperature_regime(temp):
    if pd.isna(temp):
        return "Unknown"
    elif temp < 700:
        return "Low"
    elif temp < 850:
        return "Transition"
    else:
        return "Normal_High"

df["temperature_regime"] = df["Cyclone_Inlet_Gas_Temp"].apply(
    temperature_regime
)

In [18]:
df["temperature_regime"].value_counts()

temperature_regime
Normal_High    289297
Low             79143
Transition       7959
Unknown          1320
Name: count, dtype: int64

In [19]:
df.groupby("temperature_regime")[sensor_cols].median()

,Cyclone_Inlet_Gas_Temp,Cyclone_Gas_Outlet_Temp,Cyclone_Outlet_Gas_draft,Cyclone_cone_draft,Cyclone_Inlet_Draft,Cyclone_Material_Temp
temperature_regime,,,,,,
Low,33.66,33.51,1.52,-0.69,-1.35,33.04
Normal_High,890.60,884.94,-230.06,-211.00,-179.12,926.66
Transition,838.70,826.59,-181.75,-170.72,-152.41,859.43
Unknown,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Check missing values in numeric features
numeric_features = [
    col for col in df.columns
    if col not in ["time", "temperature_regime"]
]
missing_features = df[numeric_features].isna().sum()
missing_features[missing_features > 0].sort_values(ascending=False)

Cyclone_Material_Temp_change_60min             2275
Cyclone_Material_Temp_change_30min             2114
Cyclone_Material_Temp_change_5min              1941
Cyclone_Inlet_Draft_change_60min               1741
Cyclone_Outlet_Gas_draft_change_60min          1739
Cyclone_Gas_Outlet_Temp_change_60min           1739
Cyclone_Inlet_Gas_Temp_change_60min            1737
Cyclone_cone_draft_change_60min                1737
Temp_Material_Outlet_Diff                      1592
Cyclone_Material_Temp                          1591
Temp_Material_Inlet_Diff                       1591
Cyclone_Inlet_Draft_change_30min               1580
Cyclone_Gas_Outlet_Temp_change_30min           1578
Cyclone_Outlet_Gas_draft_change_30min          1578
Cyclone_Inlet_Gas_Temp_change_30min            1576
Cyclone_cone_draft_change_30min                1576
Cyclone_Inlet_Draft_change_5min                1404
Cyclone_Outlet_Gas_draft_change_5min           1402
Cyclone_Gas_Outlet_Temp_change_5min            1402
Cyclone_Inle

In [23]:
# Check for infinite values
import numpy as np
infinite_features = np.isinf(df[numeric_features]).sum()
infinite_features[infinite_features > 0]

Series([], dtype: int64)

In [24]:
print("Total numeric features:", len(numeric_features))
print("Rows:", len(df))

Total numeric features: 42
Rows: 377719


In [25]:
# Check how many rows have complete feature information
complete_rows = df[numeric_features].notna().all(axis=1)
print("Total rows:", len(df))
print("Complete rows:", complete_rows.sum())
print("Rows with missing features:", (~complete_rows).sum())
print("Percentage usable:", round(complete_rows.mean() * 100, 2), "%")

Total rows: 377719
Complete rows: 374761
Rows with missing features: 2958
Percentage usable: 99.22 %


In [ ]:
# Create the modeling dataset

model_df = df.loc[complete_rows].copy()
print("Modeling dataset shape:", model_df.shape)
print(model_df[["time"]].head())
print(model_df[["time"]].tail())

Modeling dataset shape: (374761, 44)
                  time
12 2017-01-01 01:00:00
13 2017-01-01 01:05:00
14 2017-01-01 01:10:00
15 2017-01-01 01:15:00
16 2017-01-01 01:20:00
                      time
377714 2020-08-07 11:55:00
377715 2020-08-07 12:00:00
377716 2020-08-07 12:05:00
377717 2020-08-07 12:10:00
377718 2020-08-07 12:15:00


In [27]:
# Correlation between engineered numeric features
feature_corr = model_df[numeric_features].corr().abs()
# Keep only the upper triangle so we don't see duplicate pairs
upper = feature_corr.where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)
# Find highly correlated feature pairs
high_corr_pairs = (
    upper.stack()
    .sort_values(ascending=False)
)
high_corr_pairs[high_corr_pairs > 0.95].head(30)

Cyclone_Gas_Outlet_Temp                      Cyclone_Gas_Outlet_Temp_rolling_mean_60min     0.998509
Cyclone_Inlet_Gas_Temp                       Cyclone_Inlet_Gas_Temp_rolling_mean_60min      0.997859
Cyclone_Outlet_Gas_draft_rolling_mean_60min  Cyclone_Inlet_Draft_rolling_mean_60min         0.996010
Cyclone_Outlet_Gas_draft                     Cyclone_Inlet_Draft                            0.995451
                                             Cyclone_Outlet_Gas_draft_rolling_mean_60min    0.992600
Cyclone_Inlet_Gas_Temp_rolling_mean_60min    Cyclone_Gas_Outlet_Temp_rolling_mean_60min     0.991922
Cyclone_Inlet_Draft                          Cyclone_Inlet_Draft_rolling_mean_60min         0.991299
Cyclone_Inlet_Gas_Temp                       Cyclone_Gas_Outlet_Temp                        0.990949
Cyclone_Gas_Outlet_Temp                      Cyclone_Inlet_Gas_Temp_rolling_mean_60min      0.990807
Cyclone_cone_draft                           Cyclone_cone_draft_rolling_mean_60min         

In [28]:
print("Number of feature pairs with correlation > 0.95:",
      (high_corr_pairs > 0.95).sum())

Number of feature pairs with correlation > 0.95: 33


In [ ]:
# Feature groups for anomaly detection

raw_features = sensor_cols

relationship_features = [
    "Temp_Inlet_Outlet_Diff",
    "Temp_Material_Inlet_Diff",
    "Temp_Material_Outlet_Diff",
    "Draft_Inlet_Outlet_Diff",
    "Draft_Cone_Inlet_Diff",
    "Draft_Cone_Outlet_Diff"
]

change_features = [
    col for col in df.columns
    if "_change_5min" in col
    or "_change_30min" in col
    or "_change_60min" in col
]

rolling_std_features = [
    col for col in df.columns
    if "_rolling_std_60min" in col
]
model_features = (
    raw_features
    + relationship_features
    + change_features
    + rolling_std_features
)
print("Number of model features:", len(model_features))
print("\nFeature groups:")
print("Raw:", len(raw_features))
print("Relationships:", len(relationship_features))
print("Changes:", len(change_features))
print("Rolling std:", len(rolling_std_features))

Number of model features: 36

Feature groups:
Raw: 6
Relationships: 6
Changes: 18
Rolling std: 6


In [30]:
# Verify that all selected features are present and complete
print("Missing values:")
print(model_df[model_features].isna().sum().sum())
print("\nInfinite values:")
print(np.isinf(model_df[model_features]).sum().sum())

Missing values:
0

Infinite values:
0


In [3]:
import pandas as pd
import numpy as np
df = pd.read_csv(
    "cleaned_sensor_data.csv",
    parse_dates=["time"]
)
df = df.sort_values("time").reset_index(drop=True)
print("Shape:", df.shape)

Shape: (377719, 7)


In [4]:
# Save the feature-engineered dataset for the next stage

df.to_csv("feature_engineered_data.csv", index=False)
print("Feature-engineered data saved successfully.")
print("Shape:", df.shape)

Feature-engineered data saved successfully.
Shape: (377719, 7)


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "cleaned_sensor_data.csv",
    parse_dates=["time"]
)

df = df.sort_values("time").reset_index(drop=True)

print("Shape:", df.shape)

Shape: (377719, 7)


In [6]:
sensor_cols = [
    "Cyclone_Inlet_Gas_Temp",
    "Cyclone_Gas_Outlet_Temp",
    "Cyclone_Outlet_Gas_draft",
    "Cyclone_cone_draft",
    "Cyclone_Inlet_Draft",
    "Cyclone_Material_Temp"
]

In [7]:
df["Temp_Inlet_Outlet_Diff"] = (
    df["Cyclone_Inlet_Gas_Temp"]
    - df["Cyclone_Gas_Outlet_Temp"]
)

df["Temp_Material_Inlet_Diff"] = (
    df["Cyclone_Material_Temp"]
    - df["Cyclone_Inlet_Gas_Temp"]
)

df["Temp_Material_Outlet_Diff"] = (
    df["Cyclone_Material_Temp"]
    - df["Cyclone_Gas_Outlet_Temp"]
)

df["Draft_Inlet_Outlet_Diff"] = (
    df["Cyclone_Inlet_Draft"]
    - df["Cyclone_Outlet_Gas_draft"]
)

df["Draft_Cone_Inlet_Diff"] = (
    df["Cyclone_cone_draft"]
    - df["Cyclone_Inlet_Draft"]
)

df["Draft_Cone_Outlet_Diff"] = (
    df["Cyclone_cone_draft"]
    - df["Cyclone_Outlet_Gas_draft"]
)

In [8]:
print(df.shape)

(377719, 13)


In [9]:
for col in sensor_cols:
    df[f"{col}_change_5min"] = df[col].diff()

for col in sensor_cols:
    df[f"{col}_change_30min"] = df[col].diff(6)

for col in sensor_cols:
    df[f"{col}_change_60min"] = df[col].diff(12)

In [10]:
print(df.shape)

(377719, 31)


In [11]:
for col in sensor_cols:
    df[f"{col}_rolling_mean_60min"] = (
        df[col].rolling(window=12, min_periods=6).mean()
    )

    df[f"{col}_rolling_std_60min"] = (
        df[col].rolling(window=12, min_periods=6).std()
    )

In [12]:
def temperature_regime(temp):
    if pd.isna(temp):
        return "Unknown"
    elif temp < 700:
        return "Low"
    elif temp < 850:
        return "Transition"
    else:
        return "Normal_High"

df["temperature_regime"] = df["Cyclone_Inlet_Gas_Temp"].apply(
    temperature_regime
)

In [13]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nChange features:",
      len([c for c in df.columns if "_change_" in c]))

print("Rolling mean features:",
      len([c for c in df.columns if "_rolling_mean_" in c]))

print("Rolling std features:",
      len([c for c in df.columns if "_rolling_std_" in c]))

print("Relationship features:",
      len([
          c for c in df.columns
          if c.endswith("_Diff")
      ]))

Rows: 377719
Columns: 44

Change features: 18
Rolling mean features: 6
Rolling std features: 6
Relationship features: 6


In [14]:
df.to_csv("feature_engineered_data.csv", index=False)

print("Feature-engineered data saved successfully.")
print("Shape:", df.shape)

Feature-engineered data saved successfully.
Shape: (377719, 44)
